In [181]:
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

In [182]:
# Set up Column correlation 
from sklearn.feature_selection import mutual_info_classif


def topk_corr(X: pd.DataFrame, A: str, k=100, stop_threshold=0.08, method="pearson") -> pd.Series:
    y = X[A]

    Xn = X.select_dtypes(include="number")

    Xn = Xn.loc[:, Xn.nunique(dropna=True) > 1]
    corrs = Xn.corrwith(y, method=method).dropna()
    top = corrs.abs().sort_values(ascending=False).head(k)
    return top[top >= stop_threshold]

In [183]:
# Define metamorphic transformations
def perturbation(df, cols):
    df_perturbed = df.copy()
    for col in cols:
        df_perturbed[col] = df[col] + np.random.normal(0, df[col].std(), size=df[col].shape)
    return df_perturbed

def permutation(df, cols):
    df_permuted = df.copy()
    perm = np.random.permutation(len(df))
    df_permuted[cols] = df_permuted[cols].iloc[perm].to_numpy()
    return df_permuted

In [184]:
# Apply metamorphic testing
def metamorphic_test(model, X_test, y_test, metamorphic_relations, runs=10, sample=5000, debug = False):
    passed = 0
    failed = 0
    for metamorphic in metamorphic_relations:
        target_columns = metamorphic['target'](X_test)
        print(f"Test: {metamorphic['name']}")

        for transformation in metamorphic['transformations']:
            
            print(f"{transformation['name']} test")
            acc_list = []
            for run in range(runs):
    
                data_sample = X_test.sample(n = sample)
                data_labels = y_test.loc[data_sample.index]

                tranf_data = transformation['function'](data_sample, target_columns)
                tranf_labels = y_test.loc[tranf_data.index]

                predictions = model.predict(data_sample)
                accuracy = accuracy_score(data_labels, predictions)

                if not tranf_data.empty:
                # Predictions using the model
                    meta_predictions = model.predict(tranf_data)

                    # Calculate accuracy for this partition
                    meta_accuracy = accuracy_score(tranf_labels, meta_predictions)
                    acc_list.append(abs(meta_accuracy - accuracy))

            mean_diff = np.mean(acc_list)
            if debug:
                print(f"Mean diff: {mean_diff}")
            if mean_diff < transformation['threshold']:
                print("PASSED ✅")
                passed += 1
            else: 
                print("FAILED ❌")
                failed += 1
            print()
    return passed, failed

In [185]:
# Apply equivalent partitioning
def equivalence_partitioning_test(model, X_test, y_test, partition_tests, runs=10, sample=5000, debug = False):
    passed = 0
    failed = 0
    for test in partition_tests:
        
        print(f'Partition on : {test["name"]}')
        
        part_means = []
        for partition in test['partitions']:

            #print(f'Partition: {partition['name']}')
            acc_list = []
            means = []
            for run in range(runs):
                
                data_sample = X_test.sample(n = sample)

                partition_data = data_sample[partition["condition"](data_sample)]
                partition_labels = y_test.loc[partition_data.index]  

                if not partition_data.empty:
                    predictions = model.predict(partition_data)

                    accuracy = accuracy_score(partition_labels, predictions)
                    acc_list.append(accuracy)
                    means.append(np.mean(predictions))

            #print(f"Average accuracy: {np.mean(acc_list)}")
            part_means.append(np.mean(means))
            #print(f"Deviation in accuracy: {np.std(acc_list)}")
        mean = sum(part_means)/len(part_means)
        max_diff = abs(max(part_means)-min(part_means))
        if debug:
            print(f"Max diff: {max_diff}")
        if max_diff < test['threshold']:
            passed +=1
            print("PASSED ✅")
        else:
            failed +=1 
            print("FAILED ❌")
        print()
    return passed, failed

In [186]:
# Classic ML performance
from sklearn.metrics import classification_report, confusion_matrix

def ML_evaluation(model, X_test, y_test, runs=5, sample=1000):

    print("Classic ML Evaluation")
    acc_list = []
    reports = []
    confusion_matrices = []
    for run in range(runs):
        
        data_sample = X_test.sample(n = sample)

        partition_labels = y_test.loc[data_sample.index] 

        if not data_sample.empty:
            # Predictions using the model
            predictions = model.predict(data_sample)

            # Calculate accuracy for this partition
            accuracy = accuracy_score(partition_labels, predictions)
            acc_list.append(accuracy)
            reports.append(classification_report(partition_labels, predictions, output_dict=True))
            confusion_matrices.append(confusion_matrix(partition_labels, predictions))

    print("Final Results")
    print(f"Average accuracy: {np.mean(acc_list)}")

    weighted_precision = [r['weighted avg']['precision'] for r in reports]
    weighted_recall = [r['weighted avg']['recall'] for r in reports]

    print("Average Weighted Precision:", np.mean(weighted_precision))
    print("Average Weighted Recall:", np.mean(weighted_recall))
    
    stacked = np.stack(confusion_matrices, axis=0)
    print(f"Average confusion:\n {np.mean(stacked, axis=0)}")

In [187]:
# Set up metamorphic relationships
metamorphic_relations=[
    {'name': 'age', 'target': lambda df: topk_corr(df, "person_age_at_investigation").index.tolist(),
      'transformations': [{'name': 'permutation', 'function': permutation, 'threshold': 0.02}, {'name': 'perturbation', 'function': perturbation, 'threshold': 0.02}]},
    {'name': 'language', 'target': lambda df: topk_corr(df, "personal_qualities_days_since_language_requirement").index.tolist(),
      'transformations': [{'name': 'permutation', 'function': permutation, 'threshold': 0.02}, {'name': 'perturbation', 'function': perturbation, 'threshold': 0.01}]},
    {'name': 'medical', 'target': lambda df: topk_corr(df, "exemption_days_hist_due to_your_medical_conditions").index.tolist(),
      'transformations': [{'name': 'permutation', 'function': permutation, 'threshold': 0.02}, {'name': 'perturbation', 'function': perturbation, 'threshold': 0.02}]},
    {'name': 'gender', 'target': lambda df: topk_corr(df, "relationship_child_current_number").index.tolist(),
      'transformations': [{'name': 'permutation', 'function': permutation, 'threshold': 0.02}]},
    {'name': 'social', 'target': lambda df: topk_corr(df, "pla_history_social_effort").index.tolist(),
      'transformations': [{'name': 'permutation', 'function': permutation, 'threshold': 0.02}]},
    {'name': 'gender', 'target': lambda df: topk_corr(df, "person_gender_woman").index.tolist(),
      'transformations': [{'name': 'permutation', 'function': permutation, 'threshold': 0.02}]},
]


In [188]:
# Set up equivalent partitions

def quartile_partition(X, col, quart: int):
    if quart not in [0,1,2,3]:
        raise Exception('quart needs to be a quartile')
    match quart:
        case 0: return X[col] < X[col].quantile(0.25)
        case 1: return X[col].between(X[col].quantile(0.25), X[col].quantile(0.5), inclusive = 'left')
        case 2: return X[col].between(X[col].quantile(0.5), X[col].quantile(0.75), inclusive = 'left')
        case 3: return X[col] >= X[col].quantile(0.75)

partition_tests = [
    {
    # Related to obstacles
    'name': 'age',
    'partitions': [ 
        {"name": "q0", "condition": lambda df: quartile_partition(df, 'person_age_at_investigation', 0)},
        {"name": "q1", "condition": lambda df: quartile_partition(df, 'person_age_at_investigation', 1)},
        {"name": "q2", "condition": lambda df: quartile_partition(df, 'person_age_at_investigation', 2)},
        {"name": "q3", "condition": lambda df: quartile_partition(df, 'person_age_at_investigation', 3)},
    ],
    "threshold": 0.05},{
    'name': 'language',
    'partitions': [
        {"name": "q0", "condition": lambda df: quartile_partition(df, 'personal_qualities_days_since_language_requirement', 0)},
        {"name": "q1", "condition": lambda df: quartile_partition(df, 'personal_qualities_days_since_language_requirement', 1)},
        {"name": "q2", "condition": lambda df: quartile_partition(df, 'personal_qualities_days_since_language_requirement', 2)},
        {"name": "q3", "condition": lambda df: quartile_partition(df, 'personal_qualities_days_since_language_requirement', 3)},
    ],
    "threshold": 0.05},{
    'name': 'medical',
    'partitions': [
        {"name": "q0", "condition": lambda df: quartile_partition(df, 'exemption_days_hist_due to_your_medical_conditions', 0)},
        {"name": "q1", "condition": lambda df: quartile_partition(df, 'exemption_days_hist_due to_your_medical_conditions', 1)},
        {"name": "q2", "condition": lambda df: quartile_partition(df, 'exemption_days_hist_due to_your_medical_conditions', 2)},
        {"name": "q3", "condition": lambda df: quartile_partition(df, 'exemption_days_hist_due to_your_medical_conditions', 3)},
    ],
    "threshold": 0.01},{
    'name': 'children',
    'partitions': [
        {"name": "0", "condition": lambda df: df['relationship_child_current_number'] == 0},
        {"name": "1", "condition": lambda df: df['relationship_child_current_number'] == 1},
        {"name": "2", "condition": lambda df: df['relationship_child_current_number'] == 2},
        {"name": "3", "condition": lambda df: df['relationship_child_current_number'] == 3},
    ],
    "threshold": 0.02},{
    'name': 'social',
    'partitions': [
        {"name": "social_0", "condition": lambda df: df['pla_history_social_effort'] < 0.5},
        {"name": "social_1", "condition": lambda df: df['pla_history_social_effort'] >= 0.5}
    ],
    "threshold": 0.02},{
    'name': 'gender',
    'partitions': [ 
        {"name": "m", "condition": lambda df: df['person_gender_woman'] < 0.5},
        {"name": "f", "condition": lambda df: df['person_gender_woman'] >= 0.5}
    ],
    "threshold": 0.01}
]

In [189]:
# Let's load the dataset
data = pd.read_csv('../data/synth_data_for_testing.csv')
details = pd.read_csv('../data/data_description.csv', encoding='latin1')

# Rename columns to english for ease of use
data.rename(columns=dict(zip(details['Feature (nl)'], details['Feature (en)'])), inplace=True)
#data.drop(columns=["Ja", "Nee"], inplace=True)
data.fillna(data.mean(), inplace=True)

# Let's specify the features and the target
y_test = data['checked'].astype(int)
X_test = data.drop(['checked'], axis=1)
X_test = X_test.astype(np.float32)

In [190]:
# Wrap session in .predict() model
class OnnxWrapper():
    def __init__(self, session):
        self.session = session
    def predict(self, X):
        return self.session.run(None, {'X': X.values.astype(np.float32)})[0]

In [191]:
# Let's load the model
import onnxruntime as rt

m1 = OnnxWrapper(rt.InferenceSession("../model/model_1.onnx"))
m2 = OnnxWrapper(rt.InferenceSession("../model/model_2.onnx"))

In [192]:
ML_evaluation(bad_model, X_test, y_test)

Classic ML Evaluation
Final Results
Average accuracy: 0.9362
Average Weighted Precision: 0.9305515787690378
Average Weighted Recall: 0.9362
Average confusion:
 [[885.2  14. ]
 [ 49.8  51. ]]


In [193]:
ML_evaluation(good_model, X_test, y_test)

Classic ML Evaluation
Final Results
Average accuracy: 0.8994
Average Weighted Precision: 0.8651035682683658
Average Weighted Recall: 0.8994
Average confusion:
 [[895.2   4. ]
 [ 96.6   4.2]]


In [194]:
# Debug EP
b_e_passed, b_e_failed = equivalence_partitioning_test(m1, X_test, y_test, partition_tests, debug=True);
print("m2")
b_e_passed, b_e_failed = equivalence_partitioning_test(m2, X_test, y_test, partition_tests, debug=True);

Partition on : age
Max diff: 0.17985330102942385
FAILED ❌

Partition on : language
Max diff: 0.08714026574794032
FAILED ❌

Partition on : medical
Max diff: 0.08297091242866342
FAILED ❌

Partition on : children
Max diff: 0.05373399091910456
FAILED ❌

Partition on : social
Max diff: 0.03717746151886056
FAILED ❌

Partition on : gender
Max diff: 0.01664137080656064
FAILED ❌

m2
Partition on : age
Max diff: 0.009916882332770015
PASSED ✅

Partition on : language
Max diff: 0.020376693273997513
PASSED ✅

Partition on : medical
Max diff: 0.0030391903661071418
PASSED ✅

Partition on : children
Max diff: 0.016297572351638617
PASSED ✅

Partition on : social
Max diff: 0.010983563151641923
PASSED ✅

Partition on : gender
Max diff: 0.00048455609604211754
PASSED ✅



In [195]:
# Debug MT
b_e_passed, b_e_failed = metamorphic_test(m1, X_test, y_test, metamorphic_relations, debug=True);
print("m2")
b_e_passed, b_e_failed = metamorphic_test(m2, X_test, y_test, metamorphic_relations, debug=True);

Test: age
permutation test
Mean diff: 0.057020000000000015
FAILED ❌

perturbation test
Mean diff: 0.054880000000000005
FAILED ❌

Test: language
permutation test
Mean diff: 0.05304
FAILED ❌

perturbation test
Mean diff: 0.017379999999999996
FAILED ❌

Test: medical
permutation test
Mean diff: 0.05357999999999998
FAILED ❌

perturbation test
Mean diff: 0.04958000000000001
FAILED ❌

Test: gender
permutation test
Mean diff: 0.03962000000000001
FAILED ❌

Test: social
permutation test
Mean diff: 0.04242000000000001
FAILED ❌

Test: gender
permutation test
Mean diff: 0.042840000000000024
FAILED ❌

m2
Test: age
permutation test
Mean diff: 0.00038000000000000257
PASSED ✅

perturbation test
Mean diff: 0.00015999999999999348
PASSED ✅

Test: language
permutation test
Mean diff: 0.005180000000000018
PASSED ✅

perturbation test
Mean diff: 0.002739999999999998
PASSED ✅

Test: medical
permutation test
Mean diff: 0.00041999999999997595
PASSED ✅

perturbation test
Mean diff: 0.00033999999999998474
PASSED ✅

In [196]:
# Fairness testing for m1
b_m_passed, b_m_failed = metamorphic_test(bad_model, X_test, y_test, metamorphic_relations)
b_e_passed, b_e_failed = equivalence_partitioning_test(bad_model, X_test, y_test, partition_tests);
b_t_passed = b_m_passed + b_e_passed
b_t_failed = b_m_failed + b_e_failed
b_total = b_t_passed + b_t_failed
print(f"Passed {b_t_passed}/{b_total}")
print(f"Failed {b_t_failed}/{b_total}")

Test: age
permutation test
FAILED ❌

perturbation test
FAILED ❌

Test: language
permutation test
FAILED ❌

perturbation test
FAILED ❌

Test: medical
permutation test
FAILED ❌

perturbation test
FAILED ❌

Test: gender
permutation test
FAILED ❌

Test: social
permutation test
FAILED ❌

Test: gender
permutation test
FAILED ❌

Partition on : age
FAILED ❌

Partition on : language
FAILED ❌

Partition on : medical
FAILED ❌

Partition on : children
FAILED ❌

Partition on : social
FAILED ❌

Partition on : gender
FAILED ❌

Passed 0/15
Failed 15/15


In [197]:
# Fairness testing for m2
g_m_passed, g_m_failed = metamorphic_test(good_model, X_test, y_test, metamorphic_relations)
g_e_passed, g_e_failed = equivalence_partitioning_test(good_model, X_test, y_test, partition_tests);
g_t_passed = g_m_passed + g_e_passed
g_t_failed = g_m_failed + g_e_failed
g_total = g_t_passed + g_t_failed
print(f"Passed {g_t_passed}/{g_total}")
print(f"Failed {g_t_failed}/{g_total}")

Test: age
permutation test
PASSED ✅

perturbation test
PASSED ✅

Test: language
permutation test
PASSED ✅

perturbation test
PASSED ✅

Test: medical
permutation test
PASSED ✅

perturbation test
PASSED ✅

Test: gender
permutation test
PASSED ✅

Test: social
permutation test
PASSED ✅

Test: gender
permutation test
PASSED ✅

Partition on : age
PASSED ✅

Partition on : language
PASSED ✅

Partition on : medical
PASSED ✅

Partition on : children
PASSED ✅

Partition on : social
PASSED ✅

Partition on : gender
PASSED ✅

Passed 15/15
Failed 0/15


In [198]:
print(f"Model 1 passed {b_t_passed}/{b_total}")
print(f"Model 2 passed {g_t_passed}/{g_total}")

Model 1 passed 0/15
Model 2 passed 15/15
